# Level and Structure Controls

`harmonic_content_analysis.ipynb` reports that bending removes almost all energy above 5 kHz. Two
readings of that result have to be excluded before it can be attributed to the technique.

The first is playing level. Bent notes are quieter, and in wind instruments the high frequency
content of the radiated sound generally grows with blowing pressure, so a suppression measured
between conditions that differ in level may be a level effect wearing the costume of a technique
effect.

The second is spectral structure. The harmonic component is separated from the noise floor by a
running median along frequency, and that procedure will happily report reed rattle or a room mode as
harmonic if either happens to sit above the floor.

This notebook tests both. The gain was set once before the first take and left untouched for the
whole session, so the root-mean-square level of a recording is a usable proxy for how loudly the note
was played.

In [ ]:
import csv
import wave
import numpy as np
import statistics as st

SR, FRAME_SIZE, HOP_SIZE = 48000, 1024, 256
WINDOW = np.hamming(FRAME_SIZE)
FREQ = np.fft.rfftfreq(FRAME_SIZE, 1 / SR)
BIN_WIDTH = FREQ[1]

AUDIO_DIR = '../data/audios_v2'
DESCRIPTORS = '../data/descriptors_v2.csv'


def read_wav(path):
    with wave.open(path, 'rb') as w:
        sr, width, channels = w.getframerate(), w.getsampwidth(), w.getnchannels()
        raw = w.readframes(w.getnframes())
    if width == 3:
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        x = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)
        x = np.where(x & 0x800000, x - 0x1000000, x).astype(np.float64) / 2 ** 23
    elif width == 2:
        x = np.frombuffer(raw, dtype='<i2').astype(np.float64) / 2 ** 15
    else:
        x = np.frombuffer(raw, dtype='<i4').astype(np.float64) / 2 ** 31
    if channels > 1:
        x = x.reshape(-1, channels).mean(axis=1)
    return sr, x


def analysis_window(x, start=2.0, end=5.0):
    return x[int(start * SR):int(end * SR)]


def stft_magnitude(segment):
    starts = range(0, len(segment) - FRAME_SIZE + 1, HOP_SIZE)
    return np.array([np.abs(np.fft.rfft(segment[i:i + FRAME_SIZE] * WINDOW)) for i in starts])


rows = list(csv.DictReader(open(DESCRIPTORS)))
for r in rows:
    r['f0'] = float(r['f0'])
    path = (f"{AUDIO_DIR}/harp{r['harp']}_{r['material']}_"
            f"{r['method']}_{r['note']}_rep{r['rep']}.wav")
    _, x = read_wav(path)
    segment = analysis_window(x)
    mag = stft_magnitude(segment)
    r['level_db'] = float(20 * np.log10(np.sqrt((segment ** 2).mean())))
    above = FREQ >= 5000
    r['frac_above_5k'] = float((mag[:, above] ** 2).sum() / (mag ** 2).sum())
    r['peak'] = float(np.abs(segment).max())
    r['_mag'] = mag

natural = [r for r in rows if r['method'] == 'natural']
bent = [r for r in rows if r['method'] == 'bend']
print(f'{len(natural)} natural, {len(bent)} bent')

## Level

Two questions, in increasing order of strength. Does the association between level and high frequency
content, measured *within* the natural condition, run in the direction the objection assumes? And do
the two conditions still differ once compared at matched level?

The second question is answerable without extrapolation, because the level ranges overlap.

In [ ]:
level = lambda g: [r['level_db'] for r in g]
frac = lambda g: [r['frac_above_5k'] for r in g]

print(f'natural level: median {st.median(level(natural)):.1f} dB, '
      f'range [{min(level(natural)):.1f}, {max(level(natural)):.1f}]')
print(f'bent level:    median {st.median(level(bent)):.1f} dB, '
      f'range [{min(level(bent)):.1f}, {max(level(bent)):.1f}]')

slope, intercept = np.polyfit(level(natural), np.log10(frac(natural)), 1)
r2 = np.corrcoef(level(natural), np.log10(frac(natural)))[0, 1] ** 2
print(f'\nwithin natural: log10(fraction) = {intercept:.3f} + {slope:.4f} * level_dB, R2 = {r2:.3f}')
print('a negative slope means quieter recordings carry MORE high frequency energy')

lo, hi = min(level(bent)), max(level(bent))
matched = [r for r in natural if lo <= r['level_db'] <= hi]
print(f'\nnatural recordings inside the bent level range: {len(matched)} of {len(natural)}')
print(f'  matched natural: {st.median(frac(matched)) * 100:.2f}% above 5 kHz')
print(f'  bent:            {st.median(frac(bent)) * 100:.2f}%')
print(f'  factor at matched level: {st.median(frac(matched)) / st.median(frac(bent)):.0f}x')
print(f'  factor without matching: {st.median(frac(natural)) / st.median(frac(bent)):.0f}x')

print(f'\noverlap between the two distributions:')
print(f'  weakest natural {min(frac(natural)) * 100:.2f}%, strongest bent {max(frac(bent)) * 100:.2f}%')

## Harmonic structure above 5 kHz

If the energy above 5 kHz is genuine harmonic content, it concentrates at integer multiples of the
fundamental. If it is rattle or a room mode, it does not. Comparing the energy at the harmonic bins
against the energy midway between them, over the same number of bins, gives a balanced test: zero
decibels would mean no harmonic structure at all.

In [ ]:
def harmonic_contrast_db(mag, f0, floor=5000, half_width=1):
    power = (mag ** 2).mean(axis=0)
    on = off = 0.0
    for k in range(1, int((SR / 2) / f0)):
        peak = int(round(k * f0 / BIN_WIDTH))
        between = int(round((k + 0.5) * f0 / BIN_WIDTH))
        if FREQ[peak] < floor or between + half_width >= len(power):
            continue
        on += power[peak - half_width:peak + half_width + 1].sum()
        off += power[between - half_width:between + half_width + 1].sum()
    return 10 * np.log10(on / off) if off > 0 else np.nan


for name, group in (('natural', natural), ('bent', bent)):
    v = [harmonic_contrast_db(r['_mag'], r['f0']) for r in group]
    v = [z for z in v if not np.isnan(z)]
    print(f'{name:8} median {st.median(v):5.1f} dB, range [{min(v):.1f}, {max(v):.1f}]')

## Clipping

The remaining alternative is that the high frequency content of natural notes is distortion produced
when the interface gain was set against the loudest note of the protocol.

In [ ]:
for name, group in (('natural', natural), ('bent', bent)):
    p = [r['peak'] for r in group]
    print(f'{name:8} peak median {np.median(p):.3f}, max {max(p):.3f}, '
          f'above 0.99: {sum(z > 0.99 for z in p)}')

## Shape of the difference

A purely passive low-pass loading would attenuate the upper partials and leave the lower ones
untouched, so the bent-minus-natural difference, taken relative to each recording's own fundamental,
would be negative everywhere and would fall monotonically. Computing that profile over the
within-plate contrast, where reed plate, hole and airflow direction are held fixed, shows something
else: the difference is zero or positive over the first few harmonics and turns sharply negative
beyond them. Attenuation alone does not produce gain at low harmonics.

In [ ]:
def harmonic_profile_db(group, n_harmonics=40):
    out = []
    for k in range(1, n_harmonics + 1):
        vals = []
        for r in group:
            f0, mag = r['f0'], r['_mag']
            first = int(round(f0 / BIN_WIDTH))
            kth = int(round(k * f0 / BIN_WIDTH))
            if kth + 2 >= mag.shape[1]:
                continue
            a1 = mag[:, first - 2:first + 3].max(axis=1).mean()
            ak = mag[:, kth - 2:kth + 3].max(axis=1).mean()
            if a1 > 0 and ak > 0:
                vals.append(20 * np.log10(ak / a1))
        out.append(st.median(vals) if vals else np.nan)
    return np.array(out)


select = lambda **kw: [r for r in rows if all(r[k] == v for k, v in kw.items())]
for hole, nat_note, bent_note in (('hole 2', 'F4', 'E4'), ('hole 3', 'A4', 'G4')):
    a = harmonic_profile_db(select(harp='Bb', note=nat_note, method='natural'))
    b = harmonic_profile_db(select(harp='Bb', note=bent_note, method='bend'))
    f0 = st.median([r['f0'] for r in select(harp='Bb', note=nat_note, method='natural')])
    print(f'{hole}: {nat_note} natural against {bent_note} bent')
    print(f"   {'k':>3} {'approx Hz':>10} {'natural':>9} {'bent':>9} {'difference':>11}")
    for k in (1, 2, 3, 5, 8, 12, 16, 20, 25, 30, 40):
        if k <= len(a) and not np.isnan(a[k - 1]) and not np.isnan(b[k - 1]):
            print(f'   {k:>3} {k * f0:10.0f} {a[k - 1]:9.1f} {b[k - 1]:9.1f} {b[k - 1] - a[k - 1]:+11.1f}')
    print()